In [ ]:
#Code came from Kaggle dataset generate split

import os
import shutil
import random

# Paths
base_path = os.getcwd()
dataset_path = os.path.join(base_path, "dataset")
train_path = os.path.join(base_path, "train")
val_path = os.path.join(base_path, "val")
test_path = os.path.join(base_path, "test")

# Create directories for train, val, and test splits
os.makedirs(train_path, exist_ok=True)
os.makedirs(val_path, exist_ok=True)
os.makedirs(test_path, exist_ok=True)

# Train-Val-Test split ratios
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# Iterate through class folders
for class_name in os.listdir(dataset_path):
    class_folder = os.path.join(dataset_path, class_name)
    if os.path.isdir(class_folder):  # Check if it's a directory
        # Get all image files in the class folder
        images = [os.path.join(class_folder, img) for img in os.listdir(class_folder) if img.endswith('.png')]

        # Shuffle images for randomness
        random.shuffle(images)

        # Calculate split sizes
        train_size = int(len(images) * train_ratio)
        val_size = int(len(images) * val_ratio)

        # Split images
        train_images = images[:train_size]
        val_images = images[train_size:train_size + val_size]
        test_images = images[train_size + val_size:]

        # Create class subfolders in train, val, and test directories
        os.makedirs(os.path.join(train_path, class_name), exist_ok=True)
        os.makedirs(os.path.join(val_path, class_name), exist_ok=True)
        os.makedirs(os.path.join(test_path, class_name), exist_ok=True)

        # Move images to respective directories
        for img in train_images:
            shutil.copy(img, os.path.join(train_path, class_name))
        for img in val_images:
            shutil.copy(img, os.path.join(val_path, class_name))
        for img in test_images:
            shutil.copy(img, os.path.join(test_path, class_name))

        print(f"Processed class: {class_name}")

print("Dataset split into train, val, and test folders.")

In [ ]:
#If needing to install libraries:
%pip install scikit-learn
%pip install tensorflow
%pip install matplotlib
%pip install opencv-python

In [1]:
#Libraries needed for model creation

import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from sklearn import svm, metrics
from sklearn.ensemble import RandomForestClassifier

In [3]:
#Utilizing Google Gemini to create following function
#I did not know how to split the dataset into X and y respectively

def load_split_dataset(root_dir, img_size=(128, 128)):
    # Find all unique class names across the dataset
    train_path = os.path.join(root_dir, "train")
    classes = sorted([d for d in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, d))])
    class_to_idx = {class_name: i for i, class_name in enumerate(classes)}

    def load_split(split_name):
        X = []
        y = []
        split_dir = os.path.join(root_dir, split_name)

        if not os.path.exists(split_dir):
            raise FileNotFoundError(f"Directory not found: {split_dir}")

        for class_name in classes:
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(class_dir):
                continue

            label = class_to_idx[class_name]

            # Properly indented inside the class loop
            for file_name in os.listdir(class_dir):
                if file_name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tiff")):
                    file_path = os.path.join(class_dir, file_name)
                    img = cv2.imread(file_path, cv2.IMREAD_COLOR)
                    if img is not None:
                        img = cv2.resize(img, img_size)
                        X.append(img)
                        y.append(label)

        return np.array(X), np.array(y)

    print("Loading training data...")
    X_train, y_train = load_split("train")

    print("Loading validation data...")
    X_val, y_val = load_split("val")

    print("Loading testing data...")
    X_test, y_test = load_split("test")

    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
xTrain, yTrain, xVal, yVal, xTest, yTest = load_split_dataset(base_path)

In [6]:
#Reshaping data to fit dimensions of at most 2
numFeats = xTrain.shape[1] * xTrain.shape[2] * xTrain.shape[3]
xTrainFlat = xTrain.reshape(xTrain.shape[0], numFeats)
xTestFlat = xTest.reshape(xTest.shape[0], numFeats)

In [ ]:
#Generate model using SVM
svc = svm.SVC(random_state = 42)
svc.fit(xTrainFlat, yTrain)

In [ ]:
#Generate model using RandomForest
forest = RandomForestClassifier(random_state = 42, max_depth = 20)
forest.fit(xTrainFlat, yTrain)

In [ ]:
#Generate model using CNN with no dropout layers
cnnModel = models.Sequential([
    layers.Input(shape = (128, 128, 3)),
    
    layers.Conv2D(32, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(256, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(512, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(512, activation = 'relu'),
    layers.Dense(1000, activation = 'softmax')
])

cnnModel.summary()

cnnModel.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [ ]:
#Training model on data and printing out learning curves for cnnModel with no Dropouts
from tensorflow.keras.callbacks import EarlyStopping
stoppingCond = EarlyStopping(monitor = 'val_accuracy', patience = 5, restore_best_weights = True)

cnnHistory = cnnModel.fit(xTrain, yTrain, epochs = 50, batch_size = 32, validation_data = (xVal, yVal), callbacks = [stoppingCond], verbose = 1)

In [ ]:
#General model using CNN with Dropout layers
cnnDropoutModel = models.Sequential([
    layers.Input(shape = (128, 128, 3)),
    
    layers.Conv2D(32, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Conv2D(64, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Conv2D(128, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Conv2D(256, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Conv2D(512, (3, 3), activation = 'relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Flatten(),
    layers.Dropout(0.50),
    layers.Dense(512, activation = 'relu'),
    layers.Dense(1000, activation = 'softmax')
])

cnnDropoutModel.summary()

cnnDropoutModel.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy')

In [ ]:
#Training model on data
cnnDropoutHistory = cnnDropoutModel.fit(xTrain, yTrain, epochs = 50, batch_size = 32, validation_data = (xVal, yVal), callbacks = [stoppingCond], verbose = 1)

In [ ]:
#Generate model using ResNet50

In [ ]:
#Generate model using ResNet152

In [ ]:
#Usable variable for model scorings
figureSize = (20, 15)

In [ ]:
#SVM Model scores
ySVMPred = svc.predict(xTestFlat)
print(f"Accuracy Score of SVM: {metrics.accuracy_score(yTest, ySVMPred)}")
print(f"Precision Score of SVM: {metrics.precision_score(yTest, ySVMPred)}")
print(f"F1 Score of SVM: {metrics.f1_score(yTest, ySVMPred)}")
#Confusion Matrix
cmSVM = metrics.confusion_matrix(yTest, ySVMPred)
dispSVM = metrics.ConfusionMatrixDisplay(cmSVM)
dispSVM.plot(cmap = plt.cm.Reds)
plt.figure(figsize = figureSize)
plt.title("SVM Confusion Matrix")
plt.show()
print()

In [ ]:
#RandomForest Model Scores
yForestPred = forest.predict(xTestFlat)
print(f"Accuracy Score of Random Forest: {metrics.accuracy_score(yTest, yForestPred)}")
print(f"Precision Score of Random Forest: {metrics.precision_score(yTest, yForestPred, average = 'weighted', zero_division = 0)}")
print(f"F1 Score of Random Forest: {metrics.f1_score(yTest, yForestPred, average = 'weighted', zero_division = 0)}")
#Confusion Matrix
cmForest = metrics.confusion_matrix(yTest, yForestPred)
dispForest = metrics.ConfusionMatrixDisplay(cmForest)
dispForest.plot(cmap = plt.cm.Blues)
plt.figure(figsize = figureSize)
plt.title("Random Forest Confusion Matrix")
plt.show()
print()

In [ ]:
#CNN w/ No Dropout Model Scores
cnnTestLoss, cnnTestAcc = cnnModel.evaluate(xTest, yTest, verbose = 0)
print(f'CNN Test Loss: {cnnTestLoss:.4f}')
print(f'CNN Test Accuracy: {cnnTestAcc:.4f}')
#Plotting accuracy of CNN Model
plt.figure(figsize = (10, 5))
plt.plot(cnnHistory.history['accuracy'], label = 'Training Accuracy')
plt.plot(cnnHistory.history['val_accuracy'], label = 'Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy Curves')

In [ ]:
#CNN w/ Dropout Model Scores
cnnDropTestLoss, cnnDropTestAcc = cnnDropoutModel.evaluate(xTest, yTest, verbose = 0)
print(f'CNN Test Loss: {cnnDropTestLoss:.4f}')
print(f'CNN Test Accuracy: {cnnDropTestAcc:.4f}')
#Plotting accuracy of CNN Dropout Model
plt.figure(figsize = (10, 5))
plt.plot(cnnHistory.history['accuracy'], label = 'Training Accuracy')
plt.plot(cnnHistory.history['val_accuracy'], label = 'Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy Curves')

In [ ]:
#ResNet50 Model Scores

In [ ]:
#ResNet152 Model Scores